# Laboratory 02 — Equations of state

In this laboratory you will build the van der Waals equation of state out of two physical
corrections to the ideal gas law — molecular attraction and excluded volume — explore the
P-v-T surface those corrections bend, and check the whole story against published CO2 data
from the NIST Chemistry WebBook.

Work through it in order. Where the notebook asks you to predict, write your prediction in
the cell provided **before** running the next cell. That is not a ritual: a prediction you
have committed to is the only reliable way to discover that you were wrong.

## Model specification

| | |
|---|---|
| **System** | a fixed amount $N$ of a simple compressible substance, described by $(P, v, T)$ with $v = V/N$; the ideal gas is the special case $a=b=0$ |
| **Dynamics** | none — a static equation of state, not a trajectory to integrate |
| **Boundary** | closed; $N$ fixed, $v$ and $T$ set externally, $P$ read off the equation of state |
| **Ensemble** | not applicable — macroscopic thermodynamics |
| **Ignored** | all microscopic detail; $a$ and $b$ are phenomenological constants |
| **Valid when** | dilute classical (ideal) and moderate-density near-critical (van der Waals) regimes |
| **Failure modes** | the sub-critical $(\partial P/\partial v)_T > 0$ wiggle region (module 14); quantum-degenerate regimes (module 17) |

All the physics lives in `thermolab.gases` — open it and read it. Nothing in this course is
hidden inside a framework.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import gases
from thermolab.constants import K_B, N_A
from thermolab.validation import relative_error

# CO2's van der Waals constants, fit from its measured critical point (NIST Chemistry
# WebBook: T_c = 304.13 K, P_c = 7.3773e6 Pa) via gases.vdw_constants_from_critical -- the
# per-particle convention is explained in the module page's "A note on units" box.
CO2_T_C = 304.13  # K
CO2_P_C = 7.3773e6  # Pa
CO2_A, CO2_B = gases.vdw_constants_from_critical(CO2_T_C, CO2_P_C)
v_c, t_c, p_c = gases.vdw_critical_point(CO2_A, CO2_B)

print(f"k_B = {K_B:.6e} J/K")
print(f"CO2 a = {CO2_A:.4e} Pa m^6  (fit from its critical point)")
print(f"CO2 b = {CO2_B:.4e} m^3")
print(f"predicted (v_c, T_c, P_c) = ({v_c:.4e} m^3, {t_c:.2f} K, {p_c:.4e} Pa)")

## Predict before you calculate

Commit to an answer for each of these *before* running anything.

1. You join two identical gas bottles into one — twice the gas, in twice the volume, same
   $T$. Which of $P$, $V$, $T$, $N$, $U$ change, and which stay the same?
2. CO2 at 280 K is compressed slowly at fixed temperature. Does the pressure keep climbing,
   level off, or do something else?
3. Is there a temperature above which no pressure can liquefy a gas?
4. Two different gases at the same *reduced* temperature and pressure — same compressibility
   factor $Z$, or does $Z$ depend on which gas it is?

**Your predictions:**

1.
2.
3.
4.

## Part 1 — The ideal surface, and turning on a real gas

Build the ideal-gas P-v-T surface with `gases.pvt_surface`, then use the sliders to turn on a
fraction of CO2's $a$ and $b$ and watch the fold appear. The sliders redraw on release
(`interact_manual`), not while dragging — the surface redraw, not the physics, is the slow
part.

In [ ]:
import ipywidgets as widgets

v_grid = np.linspace(1.2 * CO2_B, 6.0 * v_c, 35)
t_grid = np.linspace(0.6 * t_c, 1.6 * t_c, 35)
ideal_v, ideal_t, ideal_p = gases.pvt_surface(v_grid, t_grid, 0.0, 0.0)


def show_surface(a_fraction=0.0, b_fraction=0.0):
    a, b = a_fraction * CO2_A, b_fraction * CO2_B
    v_mesh, t_mesh, p_mesh = gases.pvt_surface(v_grid, t_grid, a, b)

    fig = plt.figure(figsize=(7.0, 5.5))
    ax = fig.add_subplot(projection="3d")
    ax.plot_wireframe(
        ideal_v * 1e27, ideal_t, ideal_p / 1e3, color="0.7", linewidth=0.4, rstride=2, cstride=2
    )
    ax.plot_surface(
        v_mesh * 1e27, t_mesh, p_mesh / 1e3, cmap="viridis", alpha=0.9,
        rstride=1, cstride=1, linewidth=0,
    )
    ax.set_xlabel("v (1e-27 m^3)")
    ax.set_ylabel("T (K)")
    ax.set_zlabel("P (kPa)")
    ax.set_title(f"a = {a_fraction:.1f} x CO2's a, b = {b_fraction:.1f} x CO2's b "
                 "(pale wireframe: the ideal surface, a=b=0)")
    plt.show()


widgets.interact_manual(
    show_surface,
    a_fraction=widgets.FloatSlider(min=0.0, max=1.0, step=0.1, value=0.0, description="a fraction"),
    b_fraction=widgets.FloatSlider(min=0.0, max=1.0, step=0.1, value=0.0, description="b fraction"),
);

## Part 2 — The doubling experiment

Join two identical samples — $N \to 2N$, $V \to 2V$, same $T$ — and tabulate every
quantity's before/after ratio. This is the falsifying experiment for the
`doubling-doubles-everything` misconception: check which ratios come out as $2$ and which
come out as $1$ against your Part-0 prediction.

In [ ]:
n_particles, volume, temperature = 5.0e22, 2.0e-3, 300.0

v = volume / n_particles
pressure = gases.van_der_waals_pressure(v, temperature, CO2_A, CO2_B)
internal_energy = 1.5 * n_particles * K_B * temperature  # 3 dof, ideal-gas-like estimate

n_joined, v_joined = 2 * n_particles, 2 * volume
v_per_particle_joined = v_joined / n_joined
pressure_joined = gases.van_der_waals_pressure(v_per_particle_joined, temperature, CO2_A, CO2_B)
internal_energy_joined = 1.5 * n_joined * K_B * temperature

rows = [
    ("N", n_particles, n_joined),
    ("V", volume, v_joined),
    ("T", temperature, temperature),
    ("P", pressure, pressure_joined),
    ("U", internal_energy, internal_energy_joined),
]
print(f"{'quantity':<10}{'base':>16}{'joined':>16}{'ratio':>10}")
for name, base, joined in rows:
    print(f"{name:<10}{base:>16.6e}{joined:>16.6e}{joined / base:>10.4f}")

## Part 3 — The isotherm family across $T_c$

`isotherm_family` evaluates several temperatures on one $v$-grid at once. Sweep through
CO2's $T_c$ and spot the wiggle appear below it.

In [ ]:
v_grid_3 = np.linspace(1.05 * CO2_B, 4.0 * v_c, 400)
temperatures_3 = np.array([0.85, 0.95, 1.0, 1.05, 1.15, 1.3]) * t_c

pressures_3 = gases.isotherm_family(v_grid_3, temperatures_3, CO2_A, CO2_B)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for temperature, pressure in zip(temperatures_3, pressures_3, strict=True):
    ax.plot(v_grid_3 * 1e27, pressure / 1e6, label=f"T = {temperature:.0f} K")
ax.axvline(v_c * 1e27, color="crimson", ls="--", lw=1.0, label="v_c")
ax.set_xlabel("v (1e-27 m^3)")
ax.set_ylabel("P (MPa)")
ax.set_title("CO2 isotherm family across T_c")
ax.legend()
plt.tight_layout()
plt.show()

## Part 4 — Measuring $T_c$ by scanning for flatness

`vdw_critical_point` gives $T_c$ from three lines of algebra. Here we *measure* it instead,
the way an experimentalist without a closed form would: scan isotherms and find where the
flattest point on the curve (the smallest $|\partial P/\partial v|$ anywhere on it) itself
shrinks to (near) zero.

Below $T_c$ the isotherm develops a genuine loop, where $\partial P/\partial v$ crosses zero
at two *ordinary* points — an exact zero, not a flatness signal, and not something a plain
"smallest slope on the grid" search can tell apart from the real critical point. So the scan
only trusts the supercritical branch, where the isotherm stays monotonic and its flattest
point shrinks continuously toward zero as $T \to T_c^+$, and stops at the first (highest)
temperature that flattens below a small threshold.

In [ ]:
def isotherm_min_abs_slope(temperature, v_grid, a, b, h_fraction=1e-4):
    """The smallest |dP/dv| found anywhere along one isotherm, by central differences."""
    h = h_fraction * v_grid
    p_plus = gases.van_der_waals_pressure(v_grid + h, temperature, a, b)
    p_minus = gases.van_der_waals_pressure(v_grid - h, temperature, a, b)
    return float(np.min(np.abs((p_plus - p_minus) / (2.0 * h))))


# 200 temperatures x a 2000-point v-grid = 4e5 evaluations, well under a second.
temperatures_scan = np.linspace(1.3 * t_c, 0.7 * t_c, 200)  # descending: supercritical first
v_scan = np.linspace(1.2 * CO2_B, 6.0 * v_c, 2000)

characteristic_slope = p_c / v_c
threshold = 0.01 * characteristic_slope

min_slopes = np.array(
    [isotherm_min_abs_slope(T, v_scan, CO2_A, CO2_B) for T in temperatures_scan]
)
first_flat = int(np.argmax(min_slopes < threshold))
measured_t_c = temperatures_scan[first_flat]
t_c_uncertainty = 0.5 * abs(temperatures_scan[1] - temperatures_scan[0])

print(f"flatness-scan T_c = {measured_t_c:.2f} +/- {t_c_uncertainty:.2f} K")
print(f"closed-form   T_c = {t_c:.2f} K   (gases.vdw_critical_point)")
print(f"NIST          T_c = {CO2_T_C:.2f} K")
print(f"scan vs NIST relative miss: {relative_error(measured_t_c, CO2_T_C):.3%}")

## Part 5 — The NIST CO2 isotherm: computing $Z$

`data/co2-isotherm-280k.csv` holds representative published values for CO2's isothermal
pressure-volume behaviour at 280 K (see the module page's Verify section, and the CSV's own
header comment, for how it was built). Load it, compute $Z = Pv/(k_BT)$ along the isotherm,
and overlay the ideal ($Z=1$) and van der Waals predictions — the falsifier for
`ideal-gas-universal`.

In [ ]:
from pathlib import Path

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    csv_path = Path("..", "..", "..", "data", "co2-isotherm-280k.csv")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSV co-located here.
    csv_path = Path("data", "co2-isotherm-280k.csv")

nist_data = np.loadtxt(csv_path, delimiter=",", comments="#")
nist_temperature, nist_pressure, nist_vm = nist_data[:, 0], nist_data[:, 1], nist_data[:, 2]
nist_v = nist_vm / N_A  # NIST reports molar volume; the course convention is per-particle

z_measured = gases.compressibility_factor(nist_pressure, nist_v, nist_temperature)
ideal_pressure = K_B * nist_temperature / nist_v
vdw_pressure = gases.van_der_waals_pressure(nist_v, nist_temperature[0], CO2_A, CO2_B)
z_vdw = gases.compressibility_factor(vdw_pressure, nist_v, nist_temperature)

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.5))
axes[0].plot(nist_vm * 1e3, nist_pressure / 1e6, "o-", color="#2563eb", label="NIST (measured)")
axes[0].plot(nist_vm * 1e3, ideal_pressure / 1e6, "--", color="0.5", label="ideal")
axes[0].plot(nist_vm * 1e3, vdw_pressure / 1e6, ":", color="#f97316", label="van der Waals")
axes[0].set_xscale("log")
axes[0].set_xlabel("molar volume (1e-3 m^3/mol)")
axes[0].set_ylabel("P (MPa)")
axes[0].set_title("CO2 at 280 K: measured vs ideal vs van der Waals")
axes[0].legend()

axes[1].plot(nist_vm * 1e3, z_measured, "o-", color="#2563eb", label="Z, measured")
axes[1].plot(nist_vm * 1e3, z_vdw, ":", color="#f97316", label="Z, van der Waals")
axes[1].axhline(1.0, ls="--", color="0.5", label="Z = 1 (ideal)")
axes[1].set_xscale("log")
axes[1].set_xlabel("molar volume (1e-3 m^3/mol)")
axes[1].set_ylabel("Z = P v / (k_B T)")
axes[1].set_title("Compressibility factor along the 280 K isotherm")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Z ranges from {z_measured.max():.3f} (dilute) to {z_measured.min():.3f} (compressed "
      "liquid) -- nowhere near the ideal-gas value of 1 once compression is well under way.")

## Part 6 — Corresponding states: CO2, N2 and argon collapse

CO2, N2 and argon have wildly different $a$ and $b$ — and wildly different critical points.
Using only their published critical constants (no isotherm data needed for N2 or argon), plot
each gas's isotherm at the *same* reduced temperature $T_r$ and watch them collapse onto one
curve in reduced variables, exactly as `vdw_pressure_reduced` predicts with no substance-
specific input at all.

In [ ]:
species_critical_points = {
    "CO2": (304.13, 7.3773e6),
    "N2": (126.19, 3.3958e6),
    "Ar": (150.9, 4.87e6),
}

v_r_grid = np.linspace(0.5, 4.0, 300)
t_r_shared = 1.1  # the same reduced temperature for every gas

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for name, (t_c_i, p_c_i) in species_critical_points.items():
    a_i, b_i = gases.vdw_constants_from_critical(t_c_i, p_c_i)
    v_c_i, _t_c_i, _p_c_i = gases.vdw_critical_point(a_i, b_i)
    v_i = v_r_grid * v_c_i
    temperature_i = t_r_shared * t_c_i
    pressure_i = gases.van_der_waals_pressure(v_i, temperature_i, a_i, b_i)
    p_r_i, v_r_i, _t_r_i = gases.reduced_variables(pressure_i, v_i, temperature_i, a_i, b_i)
    ax.plot(v_r_i, p_r_i, lw=2.5, alpha=0.8, label=name)

p_r_universal = gases.vdw_pressure_reduced(v_r_grid, t_r_shared)
ax.plot(v_r_grid, p_r_universal, "k--", lw=1.2, label="vdw_pressure_reduced (parameter-free)")
ax.set_xlabel("v_r = v / v_c")
ax.set_ylabel("P_r = P / P_c")
ax.set_title(f"Corresponding states at T_r = {t_r_shared}: three gases, one curve")
ax.legend()
plt.tight_layout()
plt.show()

## Part 7 — Automated checks

A handful of the assertions this notebook's story depends on, mirrored from the project's
test suite.

In [ ]:
from thermolab import kinetics, paths

# 1. The ideal-gas limit: a = b = 0 matches gases.ideal_gas_pressure exactly.
ideal_ref = gases.ideal_gas_pressure(1000, 300.0, 1e-3)
vdw_zero = gases.van_der_waals_pressure(1e-6, 300.0, 0.0, 0.0)
assert relative_error(float(vdw_zero), K_B * 300.0 / 1e-6) < 1e-12

# 2. The closed-form critical point matches the pressure formula evaluated there.
p_at_critical = gases.van_der_waals_pressure(v_c, t_c, CO2_A, CO2_B)
assert relative_error(float(p_at_critical), p_c) < 1e-10

# 3. Z_c = 3/8, exactly, for every van der Waals substance.
z_c = gases.compressibility_factor(p_c, v_c, t_c)
assert relative_error(float(z_c), 3.0 / 8.0) < 1e-12

# 4. The C4a dedup: kinetics.py and paths.py re-import gases.py's functions by identity.
assert kinetics.ideal_gas_pressure is gases.ideal_gas_pressure
assert paths.ideal_gas_pressure is gases.ideal_gas_pressure

print("all checks passed")

## Part 8 — Measurement: quoting $T_c$ with its uncertainty

A number without an uncertainty is not a measurement. Part 4's flatness scan gives one; this
cell states the result the way an experimental report would.

In [ ]:
agrees = abs(measured_t_c - CO2_T_C) <= 3.0 * t_c_uncertainty

print(f"T_c(CO2) = {measured_t_c:.2f} +/- {t_c_uncertainty:.2f} K   (flatness scan, this notebook)")
print(f"T_c(CO2) = {CO2_T_C:.2f} K                     (NIST Chemistry WebBook, reference)")
print(f"T_c(CO2) = {t_c:.2f} K                     (gases.vdw_critical_point, closed form)")
print()
print(f"The scan's own uncertainty {'comfortably covers' if agrees else 'does not cover'} "
      "the gap to the NIST reference -- refining the temperature grid (more than 200 points) "
      "and tightening the flatness threshold would shrink that uncertainty further, the same "
      "convergence behaviour the project's convergence tests check for the closed-form "
      "derivative directly.")

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "02-equations-of-state.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. A student says: "Real gases deviate from the ideal gas law only because their molecules
   take up space." Improve this sentence so that it is actually correct, and say precisely
   what it leaves out.
2. Explain, without equations, why doubling both the amount of a real gas and its container's
   volume, at the same temperature, leaves its pressure unchanged.
3. What does today's flatness-scan measurement of $T_c$ *not* establish about the van der
   Waals model near the critical point?

**Your answers:**

1.
2.
3.